Adjust import path:


In [1]:
import sys
from pathlib import Path

repo_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "ptps_wildfire_demo").is_dir()
)
sys.path.insert(0, str(repo_root))

In [2]:
import httpx

from ptps_wildfire_demo.proxy.resolver import Resolver

client = httpx.AsyncClient()
resolver = Resolver(client)

In [3]:
# make this repeatable
SEED = 1

sample = resolver.drp_rescues.copy()

# some URLs are missing the protocol
is_http = sample["data_source"].str.startswith("http://")
is_https = sample["data_source"].str.startswith("https://")
sample.loc[~(is_http | is_https), "data_source"] = "https://" + sample["data_source"]

# get rid of duplicate URLs
sample = sample.drop_duplicates("data_source")
sample = sample.sample(100, random_state=SEED)
sample = sample.sort_values("data_source")

sample

,title,organization,agency,description,data_source,dataset_source_status,websites,metadata_available,metadata_url,category,last_modified,url,resources
788,Data and code from - The Impacts of Parental C...,National Agricultural Library,U.S. Department of Agriculture,<NA>,https://agdatacommons.nal.usda.gov/articles/da...,<NA>,agdatacommons.nal.usda.gov,True,http://web.archive.org/web/20251205201406/http...,"[Agriculture, Science & Research]",2026-07-23,/datasets/data-and-code-from---the-impacts-of-...,"[{'id': 3476, 'status': 'Finished', 'download_..."
895,Data from - Deer keds and blacklegged ticks in...,National Agricultural Library,U.S. Department of Agriculture,<NA>,https://agdatacommons.nal.usda.gov/articles/da...,<NA>,agdatacommons.nal.usda.gov,True,http://web.archive.org/web/20251205051511/http...,"[Agriculture, Science & Research]",2026-07-27,/datasets/data-from---deer-keds-and-blacklegge...,"[{'id': 3705, 'status': 'Finished', 'download_..."
942,Data from - Full-length 16S rRNA sequencing on...,National Agricultural Library,U.S. Department of Agriculture,<NA>,https://agdatacommons.nal.usda.gov/articles/da...,<NA>,agdatacommons.nal.usda.gov,True,https://web.archive.org/web/20251108090917/htt...,"[Agriculture, Science & Research]",2026-07-29,/datasets/data-from---full-length-16s-rrna-seq...,"[{'id': 3917, 'status': 'Finished', 'download_..."
973,Data from - Honeydew associated with four comm...,National Agricultural Library,U.S. Department of Agriculture,<NA>,https://agdatacommons.nal.usda.gov/articles/da...,<NA>,agdatacommons.nal.usda.gov,True,http://web.archive.org/web/20251112201630/http...,"[Agriculture, Science & Research]",2026-07-27,/datasets/data-from---honeydew-associated-with...,"[{'id': 3786, 'status': 'Finished', 'download_..."
977,Data from - Identification and functional char...,National Agricultural Library,U.S. Department of Agriculture,<NA>,https://agdatacommons.nal.usda.gov/articles/da...,<NA>,agdatacommons.nal.usda.gov,True,http://web.archive.org/web/20250911205319/http...,"[Agriculture, Science & Research]",2026-07-20,/datasets/data-from---identification-and-funct...,"[{'id': 3330, 'status': 'Finished', 'download_..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1139,Data inputs and outputs for wildfire suppressi...,US Forest Service,U.S. Department of Agriculture,<NA>,https://www.fs.usda.gov/rds/archive/catalog/RD...,<NA>,fs.usda.gov,True,https://web.archive.org/web/20260410084938/htt...,"[Agriculture, Climate & Environment, Science &...",2026-06-29,/datasets/data-inputs-and-outputs-for-wildfire...,"[{'id': 3078, 'status': 'Finished', 'download_..."
3598,NOAA Monthly U.S. Climate Divisional Database ...,National Oceanic and Atmospheric Administration,Department of Commerce,<NA>,https://www.ncei.noaa.gov/data/climdiv/,<NA>,ncei.noaa.gov,True,https://www.ncei.noaa.gov/access/metadata/land...,[Climate & Environment],2025-04-02,/datasets/noaa-monthly-us-climate-divisional-d...,"[{'id': 684, 'status': 'Finished', 'download_d..."
3714,Paleoclimatology Borehole,National Oceanic and Atmospheric Administration,Department of Commerce,<NA>,https://www.ncei.noaa.gov/products/paleoclimat...,<NA>,ncei.noaa.gov,False,<NA>,[Climate & Environment],2025-03-02,/datasets/paleoclimatology-borehole/,"[{'id': 200, 'status': 'Finished', 'download_d..."
3718,Paleoclimatology Fire History,National Oceanic and Atmospheric Administration,Department of Commerce,<NA>,https://www.ncei.noaa.gov/products/paleoclimat...,<NA>,ncei.noaa.gov,False,<NA>,[Climate & Environment],2025-03-02,/datasets/paleoclimatology-fire-history/,"[{'id': 204, 'status': 'Finished', 'download_d..."


In [4]:
import asyncio

import httpx
import pandas as pd


async def get_status(url: str) -> str:
    try:
        response = await client.head(url, follow_redirects=True, timeout=20)
        status = response.status_code
        if 200 <= status < 300:
            return f"🟢 {status}"
        if 300 <= status < 400:
            return f"🟡 {status}"
        return f"🔴 {status}"
    except httpx.TimeoutException:
        return "🔴 timeout"
    except httpx.HTTPError as e:
        return f"🔴 error: {e}"


async def get_statuses() -> list[int | str]:
    return await asyncio.gather(*(get_status(url) for url in sample["data_source"]))


statuses = await get_statuses()
sample["status"] = statuses

pd.set_option("display.max_colwidth", 200)
sample[["data_source", "status"]]


,data_source,status
788,https://agdatacommons.nal.usda.gov/articles/dataset/Data_and_code_from_The_Impacts_of_Parental_Choice_and_Intrapopulation_Selection_for_Seed_Size_on_the_Uprightness_of_Progeny_Derived_from_Intersp...,🟢 202
895,https://agdatacommons.nal.usda.gov/articles/dataset/Data_from_Deer_keds_and_blacklegged_ticks_infesting_ungulates_in_the_United_States_molecular_detection_of_Bartonella_spp_Rickettsia_spp_Anaplasm...,🟢 202
942,https://agdatacommons.nal.usda.gov/articles/dataset/Data_from_Full-length_16S_rRNA_sequencing_on_target_microbe_establishment_in_laboratory_and_mass-reared_Mediterranean_fruit_fly/28514825,🟢 202
973,https://agdatacommons.nal.usda.gov/articles/dataset/Data_from_Honeydew_associated_with_four_common_crop_aphid_species_increases_longevity_of_the_parasitoid_wasp_Bracon_cephi_Hymenoptera_Braconidae...,🟢 202
977,https://agdatacommons.nal.usda.gov/articles/dataset/Data_from_Identification_and_functional_characterization_of_immunity-suppressing_candidate_effector_proteins_in_the_parasitic_weed_Phelipanche_a...,🟢 202
...,...,...
1139,https://www.fs.usda.gov/rds/archive/catalog/RDS-2026-0017,🟢 200
3598,https://www.ncei.noaa.gov/data/climdiv/,🟢 200
3714,https://www.ncei.noaa.gov/products/paleoclimatology/borehole,🔴 error: Server disconnected without sending a response.
3718,https://www.ncei.noaa.gov/products/paleoclimatology/fire-history,🔴 error: Server disconnected without sending a response.


In [5]:
sample[~sample["status"].str.startswith("🟢")][["data_source", "status"]]

,data_source,status
759,https://data.cdc.gov/Environmental-Health-Toxicology/Daily-Census-Tract-Level-PM2-5-Concentrations-2006/fpqb-s69d/about_data,🔴 404
764,https://data.cdc.gov/Environmental-Health-Toxicology/Daily-County-Level-PM2-5-Concentrations-2001-2014/qjju-smys/about_data,🔴 404
4162,https://data.cdc.gov/dataset/Science-Clips/biid-68vb/about_data,🔴 404
3714,https://www.ncei.noaa.gov/products/paleoclimatology/borehole,🔴 error: Server disconnected without sending a response.
3718,https://www.ncei.noaa.gov/products/paleoclimatology/fire-history,🔴 error: Server disconnected without sending a response.
